### Tools

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [35]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile",temperature=0)
model.invoke("How are you?")

AIMessage(content="I'm just a language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 39, 'total_tokens': 85, 'completion_time': 0.114302152, 'completion_tokens_details': None, 'prompt_time': 0.005689327, 'prompt_tokens_details': None, 'queue_time': 0.163742393, 'total_time': 0.119991479}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e9d6e-ff7b-7933-bbf3-f36d64b2254b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 46, 'total_tokens': 85})

In [36]:
from langchain.tools import tool

@tool 
def getWeather(location: str) -> str:
    """Get the weather at a location"""
    return f"Its sunny in {location} currently"

model_with_tool = model.bind_tools([getWeather])

In [37]:
resp = model_with_tool.invoke("What is the weather in Boston?")
print(resp)
for tool_call in resp.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'tool_calls': [{'id': 'pw1xfc5et', 'function': {'arguments': '{"location":"Boston"}', 'name': 'getWeather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 219, 'total_tokens': 233, 'completion_time': 0.053965859, 'completion_tokens_details': None, 'prompt_time': 0.016574045, 'prompt_tokens_details': None, 'queue_time': 0.31829752, 'total_time': 0.070539904}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e9d6f-1fb8-72d2-b523-5eeac7117ef0-0' tool_calls=[{'name': 'getWeather', 'args': {'location': 'Boston'}, 'id': 'pw1xfc5et', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 14, 'total_tokens': 233}
Tool: getWeather
Args: {'location': 'Boston'}


### Tool execution loops

In [52]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant. "
            "When a tool returns information, use that information "
            "to answer the user's question naturally. "
            "Do not say 'no response is necessary'."
        ),
    },
    {
        "role": "user",
        "content": "What is the weather in Boston?"
    }
]

# First model call
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# Execute tools
for tool_call in ai_msg.tool_calls:
    tool_result = getWeather.invoke(tool_call["args"]) # or tool_call
    messages.append(tool_result)

# Final model call
final_resp = model_with_tool.invoke(messages)

print("Final response:")
print(final_resp.content)

Final response:
The weather in Boston is currently sunny. If you're planning on going outside, you might want to wear sunglasses and sunscreen to protect yourself from the sun. Is there anything else you'd like to know about the weather in Boston or any other location?


In [34]:
messages

[{'role': 'user', 'content': 'What is the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'weq31xhkc', 'function': {'arguments': '{"location":"Boston"}', 'name': 'getWeather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 219, 'total_tokens': 233, 'completion_time': 0.051314819, 'completion_tokens_details': None, 'prompt_time': 0.014283487, 'prompt_tokens_details': None, 'queue_time': 0.160903522, 'total_time': 0.065598306}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e9d6d-7231-7e02-a7a7-5060000dcc30-0', tool_calls=[{'name': 'getWeather', 'args': {'location': 'Boston'}, 'id': 'weq31xhkc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 14, 'total_tokens': 233}),
 'Its sunny in Boston curre